# Orientation Testing - All Variations

This notebook tests multiple orientation calculation methods to determine what the visualization lines represent.
Each test saves an image with a descriptive filename.

In [1]:
import os
import numpy as np
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera, mi
from heatmap_situations import situations, Point
from typing import Tuple, List

# Parameters
K = 4   # Number of antennas
N = 36  # Number of RIS elements

# Create output folder
os.makedirs("orientation_test", exist_ok=True)

print("✓ Setup complete")

✓ Setup complete


In [2]:
# Define Actor class
class Actor:
    def __init__(self, name: str, position: Tuple[float, float, float],
                 orientation: Tuple[float, float, float], rows: int, cols: int):
        self.name = name
        self.position = position
        self.orientation = orientation
        self.rows = rows
        self.cols = cols

# Define all test orientation calculation functions
def calc_original(from_point: Point, to_point: Point) -> Tuple[float, float, float]:
    """Original: (0, angle, 0)"""
    vec = (to_point['x'] - from_point['x'], to_point['y'] - from_point['y'])
    angle = float(np.degrees(np.arctan2(vec[1], vec[0])))
    return (0.0, angle, 0.0)

def calc_test1_first_position(from_point: Point, to_point: Point) -> Tuple[float, float, float]:
    """Test 1: (angle, 0, 0) - angle in first position"""
    vec = (to_point['x'] - from_point['x'], to_point['y'] - from_point['y'])
    angle = float(np.degrees(np.arctan2(vec[1], vec[0])))
    return (angle, 0.0, 0.0)

def calc_test2_third_position(from_point: Point, to_point: Point) -> Tuple[float, float, float]:
    """Test 2: (0, 0, angle) - angle in third position"""
    vec = (to_point['x'] - from_point['x'], to_point['y'] - from_point['y'])
    angle = float(np.degrees(np.arctan2(vec[1], vec[0])))
    return (0.0, 0.0, angle)

def calc_test3_negative(from_point: Point, to_point: Point) -> Tuple[float, float, float]:
    """Test 3: (0, -angle, 0) - negative angle"""
    vec = (to_point['x'] - from_point['x'], to_point['y'] - from_point['y'])
    angle = float(np.degrees(np.arctan2(vec[1], vec[0])))
    return (0.0, -angle, 0.0)

def calc_test4_minus_90(from_point: Point, to_point: Point) -> Tuple[float, float, float]:
    """Test 4: (0, angle-90, 0) - subtract 90 degrees"""
    vec = (to_point['x'] - from_point['x'], to_point['y'] - from_point['y'])
    angle = float(np.degrees(np.arctan2(vec[1], vec[0])))
    return (0.0, angle - 90.0, 0.0)

def calc_test5_radians(from_point: Point, to_point: Point) -> Tuple[float, float, float]:
    """Test 5: (0, angle_rad, 0) - use radians instead of degrees"""
    vec = (to_point['x'] - from_point['x'], to_point['y'] - from_point['y'])
    angle = float(np.arctan2(vec[1], vec[0]))  # radians
    return (0.0, angle, 0.0)

def calc_test6_from_y_axis(from_point: Point, to_point: Point) -> Tuple[float, float, float]:
    """Test 6: (0, angle, 0) - measure from Y-axis instead of X-axis"""
    vec = (to_point['x'] - from_point['x'], to_point['y'] - from_point['y'])
    angle = float(np.degrees(np.arctan2(vec[0], vec[1])))  # swapped x,y
    return (0.0, angle, 0.0)

def calc_test7_plus_90(from_point: Point, to_point: Point) -> Tuple[float, float, float]:
    """Test 7: (0, angle+90, 0) - add 90 degrees"""
    vec = (to_point['x'] - from_point['x'], to_point['y'] - from_point['y'])
    angle = float(np.degrees(np.arctan2(vec[1], vec[0])))
    return (0.0, angle + 90.0, 0.0)

# List of all tests
tests = [
    ('original', 'Original (0, angle, 0)', calc_original),
    ('test1_first_pos', 'Test 1: (angle, 0, 0)', calc_test1_first_position),
    ('test2_third_pos', 'Test 2: (0, 0, angle)', calc_test2_third_position),
    ('test3_negative', 'Test 3: (0, -angle, 0)', calc_test3_negative),
    ('test4_minus90', 'Test 4: (0, angle-90, 0)', calc_test4_minus_90),
    ('test5_radians', 'Test 5: (0, radians, 0)', calc_test5_radians),
    ('test6_from_y', 'Test 6: From Y-axis', calc_test6_from_y_axis),
    ('test7_plus90', 'Test 7: (0, angle+90, 0)', calc_test7_plus_90),
]

print(f"✓ Defined {len(tests)} test variations")

✓ Defined 8 test variations


In [3]:
# Load Single Reflection scenario
situation = next((s for s in situations if s['simulation_name'] == 'Single Reflection'), None)

if not situation:
    print("❌ Single Reflection scenario not found!")
else:
    print(f"✓ Loaded scenario: {situation['simulation_name']}")
    print(f"  Size: {situation['width']}m × {situation['height']}m")
    print(f"  Buildings: {len(situation['buildings'])}")
    print(f"  RIS points: {len(situation['ris_points'])}")
    print(f"  Receivers: {len(situation['receivers'])}")

# Extract scenario data
tx_point = situation['transmitter']
ris_points = situation['ris_points']
receivers = situation['receivers']
buildings = situation['buildings']

print(f"\nTransmitter: ({tx_point['x']:.2f}, {tx_point['y']:.2f})")
print(f"RIS P1: ({ris_points[0]['x']:.2f}, {ris_points[0]['y']:.2f})")
print(f"Receiver R1: ({receivers[0]['x']:.2f}, {receivers[0]['y']:.2f})")
print(f"Receiver R2: ({receivers[1]['x']:.2f}, {receivers[1]['y']:.2f})")

✓ Loaded scenario: Single Reflection
  Size: 20m × 20m
  Buildings: 2
  RIS points: 1
  Receivers: 2

Transmitter: (3.00, 3.00)
RIS P1: (7.00, 9.00)
Receiver R1: (16.00, 11.00)
Receiver R2: (10.00, 18.00)


In [4]:
def calculate_ris_orientation(ris_point: Point, incident_point: Point,
                              reflected_points: List[Point],
                              calc_func) -> Tuple[float, float, float]:
    """Calculate RIS orientation using the provided calculation function."""
    vec_incident = (incident_point['x'] - ris_point['x'], incident_point['y'] - ris_point['y'])
    incident_angle = float(np.degrees(np.arctan2(vec_incident[1], vec_incident[0])))

    if len(reflected_points) == 0:
        # No reflected points, use incident + 90
        if calc_func == calc_test5_radians:
            return (0.0, np.radians(incident_angle + 90), 0.0)
        else:
            return (0.0, incident_angle + 90, 0.0)

    reflected_angles = []
    for rp in reflected_points:
        vec = (rp['x'] - ris_point['x'], rp['y'] - ris_point['y'])
        reflected_angles.append(float(np.degrees(np.arctan2(vec[1], vec[0]))))

    avg_reflected = sum(reflected_angles) / len(reflected_angles)
    incident_from_opposite = incident_angle + 180
    bisector = (incident_from_opposite + avg_reflected) / 2
    
    if calc_func == calc_test5_radians:
        return (0.0, np.radians(bisector - 90), 0.0)
    else:
        return (0.0, bisector - 90, 0.0)

def create_actors_with_calc(tx_point, ris_points, receivers, K, N, calc_func):
    """Create actors using specified calculation function."""
    # TX
    target = ris_points[0] if len(ris_points) > 0 else receivers[0]
    tx_orientation = calc_func(tx_point, target)
    tx_actor = Actor('T', (tx_point['x'], tx_point['y'], 1.5), tx_orientation, rows=1, cols=K)
    
    # RIS
    ris_actors = []
    for i, ris_point in enumerate(ris_points, 1):
        ris_orientation = calculate_ris_orientation(ris_point, tx_point, receivers, calc_func)
        ris_actor = Actor(f'P{i}', (ris_point['x'], ris_point['y'], 1.5), ris_orientation,
                         rows=int(np.sqrt(N)), cols=int(np.sqrt(N)))
        ris_actors.append(ris_actor)
    
    # RX
    rx_actors = []
    for i, receiver in enumerate(receivers, 1):
        target = ris_points[0] if len(ris_points) > 0 else tx_point
        rx_orientation = calc_func(receiver, target)
        rx_actor = Actor(f'R{i}', (receiver['x'], receiver['y'], 1.5), rx_orientation, rows=1, cols=K)
        rx_actors.append(rx_actor)
    
    return tx_actor, ris_actors, rx_actors

print("✓ Helper functions defined")

✓ Helper functions defined


In [5]:
# Run all tests and save images
scene_path = f"mesh_scene/{situation['simulation_name']}.xml"

print(f"\n{'='*80}")
print(f"RUNNING ALL ORIENTATION TESTS")
print(f"{'='*80}\n")

for test_name, test_description, calc_func in tests:
    print(f"\n{'='*80}")
    print(f"Test: {test_description}")
    print(f"{'='*80}")
    
    # Create actors with this calculation method
    tx_actor, ris_actors, rx_actors = create_actors_with_calc(
        tx_point, ris_points, receivers, K, N, calc_func
    )
    
    # Log orientations
    print(f"\nActor Orientations:")
    print(f"  TX: {tx_actor.orientation}")
    for ris_actor in ris_actors:
        print(f"  {ris_actor.name}: {ris_actor.orientation}")
    for rx_actor in rx_actors:
        print(f"  {rx_actor.name}: {rx_actor.orientation}")
    
    # Load fresh scene
    scene = load_scene(scene_path)
    scene.frequency = 3.5e9
    
    # Configure antenna arrays
    scene.tx_array = PlanarArray(
        num_rows=1, num_cols=K,
        vertical_spacing=0.5, horizontal_spacing=0.5,
        pattern="iso", polarization="V"
    )
    scene.rx_array = PlanarArray(
        num_rows=1, num_cols=K,
        vertical_spacing=0.5, horizontal_spacing=0.5,
        pattern="iso", polarization="V"
    )
    
    # Add actors
    tx = Transmitter(
        name=tx_actor.name,
        position=mi.Point3f(list(tx_actor.position)),
        orientation=mi.Point3f(list(tx_actor.orientation)),
        display_radius=0.5,
        color=[1.0, 0.0, 0.0]
    )
    scene.add(tx)
    
    for ris_actor in ris_actors:
        ris = Transmitter(
            name=ris_actor.name,
            position=mi.Point3f(list(ris_actor.position)),
            orientation=mi.Point3f(list(ris_actor.orientation)),
            display_radius=0.4,
            color=[0.0, 0.0, 1.0]
        )
        scene.add(ris)
    
    for rx_actor in rx_actors:
        rx = Receiver(
            name=rx_actor.name,
            position=mi.Point3f(list(rx_actor.position)),
            orientation=mi.Point3f(list(rx_actor.orientation)),
            display_radius=0.3,
            color=[0.0, 1.0, 0.0]
        )
        scene.add(rx)
    
    # Setup camera
    center_x = situation['width'] / 2
    center_y = situation['height'] / 2
    max_dim = max(situation['width'], situation['height'])
    camera_height = max_dim * 2.5
    camera_offset = max_dim * 0.15
    
    camera = Camera(
        position=mi.Point3f([center_x - camera_offset, center_y - camera_offset, camera_height]),
        look_at=mi.Point3f([center_x, center_y, 5.0])
    )
    
    # Render and save
    output_file = f"orientation_test/{test_name}.png"
    print(f"\nRendering to: {output_file}")
    
    try:
        scene.render_to_file(
            camera=camera,
            filename=output_file,
            resolution=(1400, 1000),
            num_samples=256
        )
        print(f"✅ Saved successfully")
    except Exception as e:
        print(f"❌ Error: {e}")
    
    # Clean up
    del scene

print(f"\n{'='*80}")
print(f"✅ ALL TESTS COMPLETE")
print(f"{'='*80}")
print(f"\nImages saved to orientation_test/ folder:")
for test_name, test_description, _ in tests:
    print(f"  - {test_name}.png: {test_description}")
print(f"\nCompare the line directions in each image to determine which calculation is correct.")


RUNNING ALL ORIENTATION TESTS


Test: Original (0, angle, 0)

Actor Orientations:
  TX: (0.0, 56.309932474020215, 0.0)
  P1: (0.0, -40.82156904143252, 0.0)
  R1: (0.0, -167.47119229084848, 0.0)
  R2: (0.0, -108.43494882292202, 0.0)

Rendering to: orientation_test/original.png
✅ Saved successfully

Test: Test 1: (angle, 0, 0)

Actor Orientations:
  TX: (56.309932474020215, 0.0, 0.0)
  P1: (0.0, -40.82156904143252, 0.0)
  R1: (-167.47119229084848, 0.0, 0.0)
  R2: (-108.43494882292202, 0.0, 0.0)

Rendering to: orientation_test/test1_first_pos.png
✅ Saved successfully

Test: Test 2: (0, 0, angle)

Actor Orientations:
  TX: (0.0, 0.0, 56.309932474020215)
  P1: (0.0, -40.82156904143252, 0.0)
  R1: (0.0, 0.0, -167.47119229084848)
  R2: (0.0, 0.0, -108.43494882292202)

Rendering to: orientation_test/test2_third_pos.png
✅ Saved successfully

Test: Test 3: (0, -angle, 0)

Actor Orientations:
  TX: (0.0, -56.309932474020215, 0.0)
  P1: (0.0, -40.82156904143252, 0.0)
  R1: (0.0, 167.4711922908484

TypeError: mitsuba.Point3f.__init__(): Item assignment failed.

## Summary

This notebook tested 8 different orientation calculation methods:

1. **original**: `(0, angle, 0)` - Current implementation
2. **test1_first_pos**: `(angle, 0, 0)` - Angle in first position
3. **test2_third_pos**: `(0, 0, angle)` - Angle in third position
4. **test3_negative**: `(0, -angle, 0)` - Negative angle
5. **test4_minus90**: `(0, angle-90, 0)` - Subtract 90°
6. **test5_radians**: `(0, radians, 0)` - Use radians
7. **test6_from_y**: `(0, angle_from_y, 0)` - Measure from Y-axis
8. **test7_plus90**: `(0, angle+90, 0)` - Add 90°

### Expected Visual Directions (Top-Down View)

Based on geometry:
- **TX (red)** at (3,3) → P1 at (7,9): Should point **UP-RIGHT** (~56° from +X)
- **P1 (blue)** at (7,9): Bisector orientation (~-41°)
- **R1 (green)** at (16,11) → P1 at (7,9): Should point **LEFT** (~-167° ≈ 180°)
- **R2 (green)** at (10,18) → P1 at (7,9): Should point **DOWN-LEFT** (~-108°)

### Next Steps

1. Compare all images in `orientation_test/` folder
2. Find which test shows lines pointing in the expected directions
3. That test reveals the correct orientation convention for Sionna visualization

**Note**: Since physics/path loss is already correct with the original implementation, the lines likely show array geometry, not boresight. The correct test will just help us understand the visualization convention.